In [ ]:
from collections import Counter
from itertools import combinations

import igraph as ig
import leidenalg
import matplotlib.patches as mpatches_base
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.path import Path

from helpers import init_mpl_fig, save_mpl_fig

In [ ]:
df = pd.read_parquet("gc_human_tweets.parquet").assign(
    hashtags=lambda df_: df_["hashtags"].apply(
        lambda x: x.tolist() if isinstance(x, np.ndarray) else x
    )
)
df

## Prep network

In [ ]:
# Add #metoo back to hashtag lists
df["hashtags_with_metoo"] = df["hashtags"].apply(
    lambda x: ["#metoo"] + list(x) if x else ["#metoo"]
)

# Get all co-occurrences
print("Calculating co-occurrences...")
cooccurrences = []
for hashtags in df["hashtags_with_metoo"]:
    if hashtags and len(hashtags) >= 2:
        cooccurrences.extend(list(combinations(sorted(hashtags), 2)))

edge_weights = Counter(cooccurrences)

# edges dataframe
MIN_COOCCURRENCE = 100

df_edges = (
    pd.DataFrame(
        [
            {"hashtag1": h1, "hashtag2": h2, "weight": count}
            for (h1, h2), count in edge_weights.items()
        ]
    )
    .sort_values("weight", ascending=False, ignore_index=True)
    .query(f"weight>={MIN_COOCCURRENCE}")
)
df_edges

In [ ]:
# Build full network
G = nx.from_pandas_edgelist(
    df_edges, source="hashtag1", target="hashtag2", edge_attr="weight"
)
print(
    f"Full network (edge weight > {MIN_COOCCURRENCE}): {G.number_of_nodes()} nodes, {G.number_of_edges()} edges"
)

# Filter to well-connected nodes
degrees = dict(G.degree(weight="weight"))
# Need these for plotting later
max_deg_full = max(degrees.values())  # Max across ALL nodes
min_deg_full = min(degrees.values())  # Min across ALL nodes

TOP_N = 60
top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:TOP_N]
G_temp = G.subgraph(top_nodes).copy()

# Remove peripheral nodes (must have degree >= threshold in subgraph)
degrees_in_subgraph = dict(G_temp.degree())
MIN_DEGREE_THRESHOLD = 10
well_connected = [
    node for node, deg in degrees_in_subgraph.items() if deg >= MIN_DEGREE_THRESHOLD
]
# Remove data artifacts
bad_nodes = ["#1uhttps"]
well_connected = [node for node in well_connected if node not in bad_nodes]

# Create final visualization graph (SKIP THE SPAM FILTER)
G_viz = G.subgraph(well_connected).copy()
print(
    f"Filtered network: {G_viz.number_of_nodes()} nodes, {G_viz.number_of_edges()} edges"
)

In [ ]:
def nx_to_igraph(G):
    """Convert NetworkX graph to igraph"""
    node_list = list(G.nodes())
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}

    edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
    weights = [G[u][v]["weight"] for u, v in G.edges()]

    g = ig.Graph(n=len(node_list), edges=edges, directed=False)
    g.vs["name"] = node_list
    g.es["weight"] = weights

    return g, node_list


# Convert to igraph
g_igraph, node_list = nx_to_igraph(G_viz)

# Run Leiden algorithm
partition = leidenalg.find_partition(
    g_igraph,
    leidenalg.ModularityVertexPartition,
    weights="weight",
    seed=42,  # For reproducibility
)

# Convert back to NetworkX format
communities = []
for community_nodes in partition:
    communities.append(set(node_list[i] for i in community_nodes))

# Map nodes to communities
node_to_comm = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_comm[node] = i

print(f"Leiden detected {len(communities)} communities")

# layout
pos = nx.kamada_kawai_layout(G_viz, weight="weight")

In [ ]:
community_colors = [
    (51 / 255, 34 / 255, 136 / 255),  # #332288 - purple
    (68 / 255, 170 / 255, 153 / 255),  # #44aa99 - teal
    (221 / 255, 204 / 255, 119 / 255),  # #ddcc77 - yellow
    (136 / 255, 34 / 255, 85 / 255),  # #882255 - magenta
]
# in case more colors are needed
while len(community_colors) < len(communities):
    community_colors.append([0.5, 0.5, 0.5])
# edge color
edge_data = []
for u, v in G_viz.edges():
    weight = G_viz[u][v]["weight"]

    if node_to_comm[u] == node_to_comm[v]:
        color = community_colors[node_to_comm[u]]
        alpha = 0.5
    else:
        color = [0.7, 0.7, 0.7]
        alpha = 0.25

    edge_data.append({"u": u, "v": v, "color": color, "alpha": alpha, "weight": weight})

# Normalize edge widths
weights = [e["weight"] for e in edge_data]
max_w, min_w = max(weights), min(weights)
for e in edge_data:
    normalized = (e["weight"] - min_w) / (max_w - min_w)
    e["width"] = 0.001 + normalized * 2.0

In [ ]:
_, ax = init_mpl_fig(aspect_ratio=(16, 12), scale=3)

# Draw curved edges
for e in edge_data:
    x0, y0 = pos[e["u"]]
    x1, y1 = pos[e["v"]]
    mid_x, mid_y = (x0 + x1) / 2, (y0 + y1) / 2
    dx, dy = x1 - x0, y1 - y0
    length = np.sqrt(dx**2 + dy**2)
    if length > 0:
        perp_x, perp_y = -dy / length, dx / length
        curve_amount = 0.3 * length
        ctrl_x = mid_x + perp_x * curve_amount
        ctrl_y = mid_y + perp_y * curve_amount
    else:
        ctrl_x, ctrl_y = mid_x, mid_y

    verts = [(x0, y0), (ctrl_x, ctrl_y), (x1, y1)]
    codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]
    path = Path(verts, codes)
    patch = mpatches_base.PathPatch(
        path,
        facecolor="none",
        edgecolor=e["color"],
        alpha=e["alpha"],
        linewidth=e["width"],
        capstyle="round",
        zorder=1,
    )
    ax.add_patch(patch)

# Draw semi-transparent circles behind nodes
for node in G_viz.nodes():
    x, y = pos[node]

    # Circle size based on degree
    degree = degrees[node]
    norm_deg = (degree - min_deg_full) / (max_deg_full - min_deg_full)

    if node == "#metoo":
        circle_size = 0.02  # Larger for metoo
        alpha = 0.3
    else:
        circle_size = 0.005 + norm_deg * 0.05  # Scale with degree
        alpha = 0.2 + norm_deg * 0.2  # More important = more opaque

    # Draw circle with community color
    circle = plt.Circle(
        (x, y),
        circle_size,
        color=community_colors[node_to_comm[node]],
        alpha=alpha,
        zorder=1.5,
    )  # Between edges (1) and text (2)
    ax.add_patch(circle)

# Draw text
for node in G_viz.nodes():
    x, y = pos[node]

    degree = degrees[node]
    norm_deg = (degree - min_deg_full) / (max_deg_full - min_deg_full)

    if node == "#metoo":
        font_size = 40
    else:
        font_size = 12 + norm_deg * 60

    ax.text(
        x,
        y,
        node.replace("#", ""),
        fontsize=font_size,
        fontweight="bold",
        ha="center",
        va="center",
        zorder=2,
    )

ax.axis("off")
ax.margins(0.0)
plt.tight_layout()
save_mpl_fig("output/figures/network_hashtags")